In [7]:
%run preprocessing.ipynb

In [8]:
import pandas as pd
import customtkinter as ctk
from PIL import Image, ImageTk
from customtkinter import filedialog
from tkinter import ttk, messagebox
import os

data = None
selected_cols = [] 

root = ctk.CTk()
ctk.set_appearance_mode("light")
root.title('Data Analysis')
root.geometry('1000x800')
root.configure(fg_color="#F8F9FA")  # Soft White Background

# function to upload and updata dataset file
def upload(place):
    global data
    path = filedialog.askopenfilename(filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx")])
    try:
        if path.endswith('.csv'):
            data = pd.read_csv(path)
        else:
            data = pd.read_excel(path)
        place.configure(text=str(path), text_color='black')

        for trees in right.winfo_children():  # clear parent frame from existing tree
            trees.destroy()
        tree = ttk.Treeview(right)  # Treeview widget to display data
        viewTables(tree, data, right)  
        object_types_to_categorical(data)
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred: {e}")
        place.configure(text='Error uploading file', text_color='red')

#function to view updates done to df after a specific cleaning operation
def update_tree_view(parent_frame):
    for widget in parent_frame.winfo_children():  # clear parent frame from the last tree/tree update
        widget.destroy()
    new_tree = ttk.Treeview(parent_frame)  # create new tree
    viewTables(new_tree, data, parent_frame)

#function to show df content
def viewTables(tree, df, parent_frame):
    try:
        tree.delete(*tree.get_children())# Clear previous data
        tree['columns'] = list(df.columns)  # Set column headers
        tree['show'] = 'headings'

        # Create columns for each data field
        for col in tree['columns']:
            tree.heading(col, text=col)
            tree.column(col, anchor='center')

        # Insert rows into the Treeview
        data_rows = df.to_numpy().tolist()
        for idx, row in enumerate(data_rows):
            tree.insert('', 'end', values=row)  # Add Index and row data

        # Create vertical scrollbar
        vsb = ttk.Scrollbar(parent_frame, orient="vertical", command=tree.yview)
        tree.configure(yscrollcommand=vsb.set)
        vsb.pack(side="right", fill="y")

        # Create horizontal scrollbar
        hsb = ttk.Scrollbar(parent_frame, orient="horizontal", command=tree.xview)
        tree.configure(xscrollcommand=hsb.set)
        hsb.pack(side="bottom", fill="x")  # Horizontal scrollbar at the bottom

        tree.pack(pady=20, fill='both', expand=True)
        treeDefaults()  # Set default style for treeview
        # Bind row click event to select a row
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred: {e}")

# function to set tree style
def treeDefaults():
    style = ttk.Style()
    style.theme_use('clam')
    style.configure("Treeview.Heading", font=("Arial", 12, "bold"), foreground="#212529", background="#E9ECEF")
    style.configure("Treeview", font=("Arial", 11), rowheight=25, background="white", fieldbackground="white", foreground="#212529")

    # Row selection color
    style.map('Treeview',
              background=[('selected', '#90CAF9')],
              foreground=[('selected', '#212529')])

#function to assign function to their designed comboboxes
#couldn't be assigned directly as data is empty initially
def excute_method(choice):
    global selected_cols
    try:
        methods = [
            numerical_missing_values,
            categorical_missing_values,
            methods_duplicates,
            methods_outliers,
            methods_encoding,
            methods_normalization
        ]
        for method_list in methods:
            for name, function in method_list:
                if name == choice:
                    if name in ['One hot encoder', 'Label encoder', 'Binary Encoder', 'Ordinal encoder']:
                        selected_cols = []  
                        cols_to_encode(data)
                    else:
                        function()
                    break
        update_tree_view(right)
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred: {e}")

#fucntions to create checkboxes for categorical columns
def cols_to_encode(df):
    global selected_cols
    for child in encoding.winfo_children():
        child.destroy()

    encoding_cols = show_categorical(df)

    def toggle(col, var):
        if var.get():
            if col not in selected_cols:
                selected_cols.append(col)
        else:
            if col in selected_cols:
                selected_cols.remove(col)

    for col in encoding_cols:
        var = ctk.BooleanVar(value=False)
        checkbox = ctk.CTkCheckBox(encoding, text=col, variable=var, command=lambda col=col, var=var: toggle(col, var))
        checkbox.pack(side='top', anchor='w')
    print(list(selected_cols))
    return list(selected_cols) # Return a copy of the list

#function to apply encoding to the selected columns
def apply_encoding(df, selected_cols):
    if preprocessing_options_3.get() == 'One hot encoder':
        one_hot_encoder(df, selected_cols)
    elif preprocessing_options_3.get() == 'Label encoder':
        label_encoder(df, selected_cols)
    elif preprocessing_options_3.get() == 'Binary Encoder':
        binary_encoding(df, selected_cols)
    elif preprocessing_options_3.get() == 'Ordinal encoder':
        ordinal_encoder(df, selected_cols)
    update_tree_view(right)
    cols_to_encode(df)

# ===== Tabs =====
tabs = ctk.CTkTabview(root, width=980, height=860, fg_color="#E9ECEF")
preprocessing = tabs.add('Preprocessing')
visualization = tabs.add('Visualization')
ml_model = tabs.add('ML model')
tabs.pack(padx=10, pady=10)

# ============================= Preprocessing Tab =============================
data_info = ctk.CTkTabview(preprocessing, width=200, fg_color='white')
info = data_info.add('Data Info')
cols = data_info.add('processing')
data_info.pack(fill='y', side='left', padx=10, pady=10)

uploadFrame = ctk.CTkFrame(preprocessing, fg_color="white", height=40)
uploadBtn = ctk.CTkButton(uploadFrame, width=100, height=30, text="Upload", fg_color="#90CAF9", hover_color="#64B5F6",
                            text_color="#212529", command=lambda: upload(header))
uploadBtn.pack(side='right', padx=10, pady=5)

uploadTxt = ctk.CTkLabel(uploadFrame, text="File Path:", font=('Arial', 13), text_color="#212529")
uploadTxt.pack(side='left', padx=5)

header = ctk.CTkLabel(uploadFrame, text="No file selected", font=('Arial', 13), anchor='w', wraplength=600, text_color="#212529")
header.pack(side='left', fill='x', expand=True, padx=5)

uploadFrame.pack(fill='x', padx=10, pady=(10, 5))

right = ctk.CTkFrame(preprocessing, fg_color='white')
right.pack(padx=10, pady=10, expand=True, fill='both')

# ComboBox Titles
comboBox_title = ['numerical missing values','categorical missing values','Handling duplicates',  'Removing outliers', 'Encoding', 'Normalization']

# Methods List (Each option list with a corresponding lambda function)
numerical_missing_values= [
    ('Simple imputer mean', lambda: simple_imputer(data, 'mean')),
    ('Simple imputer median', lambda: simple_imputer(data, 'median')),
    ('Simple imputer mode', lambda: simple_imputer(data, 'most_frequent')),
    ('KNN imputer', lambda: k_mean(data, n_value=5)),
    ('Iterative imputer', lambda: iterative_imputer(data))
]

categorical_missing_values=[
    ('Simple imputer mode', lambda: fill_categoriacl(data, 'most_frequent')),
    ('Simple imputer constant', lambda: fill_categoriacl(data, 'constant'))
]

methods_duplicates = [
    ('Handling duplicates', lambda: data.drop_duplicates(inplace=True))
]

methods_encoding = [
    ('One hot encoder', lambda: None),  
    ('Label encoder', lambda: None),    
    ('Binary Encoder', lambda: None),   
    ('Ordinal encoder', lambda: None)    
]

methods_outliers = [
    ('Z score', lambda: outliers_z_score(data)),
    ('IQR method', lambda: outliers_IQR(data))
]

methods_normalization = [
    ('Mini-max scaler', lambda: minMax_scaler(data)),
    ('Standard scaler', lambda: standard_scaler(data))
]

# Creating each combo box individually
preprocessing_options_1 = ctk.CTkComboBox(cols, values=[name for name, _ in numerical_missing_values],
                                         fg_color="white", border_color="#CED4DA", text_color="#212529",
                                         button_color="#90CAF9", button_hover_color="#64B5F6",
                                         dropdown_fg_color="white", dropdown_hover_color="#E9ECEF",
                                         dropdown_text_color="#212529", justify='left', command=excute_method)
preprocessing_options_1.set(comboBox_title[0])
preprocessing_options_1.pack(pady=10)

preprocessing_options_6 = ctk.CTkComboBox(cols, values=[name for name, _ in categorical_missing_values],
                                         fg_color="white", border_color="#CED4DA", text_color="#212529",
                                         button_color="#90CAF9", button_hover_color="#64B5F6",
                                         dropdown_fg_color="white", dropdown_hover_color="#E9ECEF",
                                         dropdown_text_color="#212529", justify='left', command=excute_method)
preprocessing_options_6.set(comboBox_title[1])
preprocessing_options_6.pack(pady=10)

preprocessing_options_2 = ctk.CTkComboBox(cols, values=[name for name, _ in methods_duplicates],
                                         fg_color="white", border_color="#CED4DA", text_color="#212529",
                                         button_color="#90CAF9", button_hover_color="#64B5F6",
                                         dropdown_fg_color="white", dropdown_hover_color="#E9ECEF",
                                         dropdown_text_color="#212529", justify='left', command=excute_method)
preprocessing_options_2.set(comboBox_title[2])
preprocessing_options_2.pack(pady=10)

preprocessing_options_4 = ctk.CTkComboBox(cols, values=[name for name, _ in methods_outliers],
                                         fg_color="white", border_color="#CED4DA", text_color="#212529",
                                         button_color="#90CAF9", button_hover_color="#64B5F6",
                                         dropdown_fg_color="white", dropdown_hover_color="#E9ECEF",
                                         dropdown_text_color="#212529", justify='left', command=excute_method)
preprocessing_options_4.set(comboBox_title[3])
preprocessing_options_4.pack(pady=10)

preprocessing_options_3 = ctk.CTkComboBox(cols, values=[name for name, _ in methods_encoding],
                                         fg_color="white", border_color="#CED4DA", text_color="#212529",
                                         button_color="#90CAF9", button_hover_color="#64B5F6",
                                         dropdown_fg_color="white", dropdown_hover_color="#E9ECEF",
                                         dropdown_text_color="#212529", justify='left', command=excute_method)
preprocessing_options_3.set(comboBox_title[4])
preprocessing_options_3.pack(pady=10)

preprocessing_options_5 = ctk.CTkComboBox(cols, values=[name for name, _ in methods_normalization],
                                         fg_color="white", border_color="#CED4DA", text_color="#212529",
                                         button_color="#90CAF9", button_hover_color="#64B5F6",
                                         dropdown_fg_color="white", dropdown_hover_color="#E9ECEF",
                                         dropdown_text_color="#212529", justify='left', command=excute_method)
preprocessing_options_5.set(comboBox_title[5])
preprocessing_options_5.pack(pady=10)

encoding_label = ctk.CTkLabel(cols, text='Encoding', font=('Arial', 16), fg_color='#F0F8FF', text_color="#212529")
encoding_label.pack()

encoding = ctk.CTkScrollableFrame(cols, orientation='vertical')
encoding.pack(side='top', fill='both')

applyBtn = ctk.CTkButton(cols, text='Apply', width=150, height=50, command=lambda: apply_encoding(data, cols_to_encode(data)))
applyBtn.pack(side='top', pady=5)

# ============================= Visualization Tab =============================
right_frame = ctk.CTkFrame(visualization, fg_color='white', width=300)
right_frame.pack(side='left', fill='y', padx=10, pady=10)

graphs = ctk.CTkFrame(right_frame, fg_color='white', width=280, height=220)
graphs.pack(side='top', fill='x', pady=10)

# a dictionary to access buttons outside the for loop
btn_ref = {}

plot_types = [
    ("Histogram", ctk.CTkImage(Image.open(os.path.join("images", "histogram.png")), size=(32, 24))),
    ("Scatter", ctk.CTkImage(Image.open(os.path.join("images", "scatter-graph.png")), size=(32, 24))),
    ("Line", ctk.CTkImage(Image.open(os.path.join("images", "line.png")), size=(32, 24))),
    ("Bar", ctk.CTkImage(Image.open(os.path.join("images", "bar.png")), size=(32, 24))),
    ("Box", ctk.CTkImage(Image.open(os.path.join("images", "plot.png")), size=(32, 24))),
    ("Violin", ctk.CTkImage(Image.open(os.path.join("images", "data-chart.png")), size=(32, 24))),
    ("Heatmap", ctk.CTkImage(Image.open(os.path.join("images", "business.png")), size=(32, 24)))
]
# for developer: add a third argument for command and do the same in plot_types
for index, (plot, image) in enumerate(plot_types):
    row = index // 3
    col = index % 3
    btn = ctk.CTkButton(graphs, text=plot, image=image, compound="top", width=85, height=65,
                        font=ctk.CTkFont(size=12), fg_color="transparent", hover_color="#64B5F6", text_color="#0D1B2A")
    btn.grid(row=row, column=col, padx=6, pady=6)
    btn_ref[plot] = btn

columns = ctk.CTkFrame(right_frame, fg_color='white', width=280)
columns.pack(side='top', fill='both', expand=True, pady=10)
columns.pack_propagate(False)

select_col = ctk.CTkLabel(columns, text='Select columns to plot', font=('Arial', 16), fg_color='#F0F8FF', text_color="#212529")
select_col.pack(side='top', fill='x', pady=5)

visualize = ctk.CTkFrame(visualization, fg_color='white')
visualize.pack(side='right', fill='both', expand=True, padx=10, pady=10)

visualize_label = ctk.CTkLabel(visualize, text='Columns Relationships', font=('Arial', 20), fg_color='#F0F8FF', text_color="#212529")
visualize_label.pack(side='top', fill='x', pady=(5, 10))

# ============================= ML Model Tab =============================

root.mainloop()

[]
['ECigaretteUsage']
['ECigaretteUsage']
[]
['State']
['State']
[]
['AgeCategory']
['AgeCategory']
